In [ ]:
!pip install clickhouse-connect pandas numpy scipy matplotlib

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
from clickhouse_connect import get_client

In [3]:
client = get_client(
    host="localhost",
    port=8123,
    username="metrics_user",
    password="metrics_pass",
    database="metrics"
)

client.command("SELECT 1")

1

In [ ]:
APP_ID = "com.ndevelop.perfx"
METRIC_ID = "frameTime"
SCREEN_NAME = "EditorScreen"

# TODO change version
BASELINE_VERSION = "1.0"
CURRENT_VERSION = "1.0"
DEVICE_COHORT = "mid_range"

In [28]:
query = f"""
SELECT
    app_version,
    value
FROM metric_records
WHERE app_id = '{APP_ID}'
AND metric_id = '{METRIC_ID}'
AND app_version IN ('{BASELINE_VERSION}', '{CURRENT_VERSION}')
"""

df = client.query_df(query)
df.head()

,app_version,value
0,1.0,16.666666
1,1.0,16.666666
2,1.0,16.666666
3,1.0,16.666666
4,1.0,16.666666


In [29]:
baseline = df[df["app_version"] == BASELINE_VERSION]["value"].astype(float).to_numpy()
current = df[df["app_version"] == CURRENT_VERSION]["value"].astype(float).to_numpy()

len(baseline), len(current)

(3915, 3915)

In [30]:
MIN_SAMPLES = 50

if len(baseline) < MIN_SAMPLES or len(current) < MIN_SAMPLES:
    print("Not enough samples")
else:
    print("Enough samples")

Enough samples


In [31]:
def summarize(x):
    return {
        "count": len(x),
        "mean": np.mean(x),
        "p50": np.percentile(x, 50),
        "p95": np.percentile(x, 95),
        "p99": np.percentile(x, 99),
    }

baseline_stats = summarize(baseline)
current_stats = summarize(current)

baseline_stats, current_stats

({'count': 3915,
  'mean': np.float64(21.590581817894304),
  'p50': np.float64(16.666666000000006),
  'p95': np.float64(22.727271818181826),
  'p99': np.float64(52.68749757250037)},
 {'count': 3915,
  'mean': np.float64(21.590581817894304),
  'p50': np.float64(16.666666000000006),
  'p95': np.float64(22.727271818181826),
  'p99': np.float64(52.68749757250037)})

In [32]:
delta_p50 = (current_stats["p50"] - baseline_stats["p50"]) / baseline_stats["p50"]
delta_p95 = (current_stats["p95"] - baseline_stats["p95"]) / baseline_stats["p95"]

print("delta_p50 =", delta_p50)
print("delta_p95 =", delta_p95)

delta_p50 = 0.0
delta_p95 = 0.0


In [33]:
stat, p_value = mannwhitneyu(current, baseline, alternative="greater")
print("p_value =", p_value)

p_value = 0.5000022153621949


In [36]:
P95_THRESHOLD = 0.20   # 20%
P50_THRESHOLD = 0.10   # 10%
P_VALUE_THRESHOLD = 0.05

is_regression = (
    p_value < P_VALUE_THRESHOLD and
    (delta_p95 > P95_THRESHOLD or delta_p50 > P50_THRESHOLD)
)

print("Regression detected:", is_regression)

Regression detected: False
